# SQL Data Extraction & KPI Engineering

Welcome to the SQL analysis section of my project. In this document, I connect to my cloud-based PostgreSQL database (hosted on Neon) to perform advanced data extraction, cleaning, and aggregation. 

The goal of these queries is to transform raw clinical and lifestyle data into actionable insights, preparing the datasets that will power my interactive Tableau dashboards. I have divided my SQL analysis into two main pillars: **Clinical Demographics** to understand my patient cohort, and **Key Performance Indicators (KPIs)** to evaluate survival and risk factors.

---

## Part 1: Clinical Demographics (TCGA Cohort)
Before diving into complex survival metrics, I need to establish a clear baseline of the clinical dataset. These queries extract the fundamental demographic distribution of the oncology patients:

* **Demo 1: Top Cancer Types:** I aggregate the total number of cases per oncological pathology to identify the most prevalent diseases in my study.
* **Demo 2: Gender Distribution by Cancer Type:** I break down the prevalence of each cancer type by biological sex to observe demographic correlations.
* **Demo 3: Age Distribution:** I use conditional logic (`CASE WHEN`) to create structured age buckets, allowing for a cleaner visualization of diagnosis age trends.

---

## Part 2: Lifestyle & Population KPIs (NHANES Cohort)
This section analyzes how daily habits and lifestyle choices impact overall patient survival, using my curated population dataset.

* **KPI 1: Activity vs. Survival:** I analyze the difference in average survival months between patients who engage in vigorous exercise and those who remain sedentary.
* **KPI 2: Lifestyle Risk Matrix:** I use advanced Window Functions (`RANK() OVER`) to create a comprehensive risk ranking, crossing smoking habits with sedentary behavior to see which combination yields the lowest survival rates.
* **KPI 3: Sedentary Behavior Impact:** I categorize the continuous variable of daily sedentary minutes into clear groups (<4 hours, 4-8 hours, >8 hours) to measure the exact threshold where inactivity becomes most detrimental to survival.

---

## Part 3: Clinical & Genomic KPIs (TCGA Cohort)
This final section dives into the complex medical data, leveraging CTEs (`WITH`) and string manipulation (`LOWER`, `LIKE`) to filter out missing data and evaluate pathological severity.

* **KPI 4: Genetic Mutation Burden vs. Relapse:** I group patients by their total mutation count to calculate if high genomic instability correlates with a shorter disease-free period (faster relapse).
* **KPI 5: Clinical Severity Matrix:** I calculate the average survival time by combining the Tumor Stage (I-IV) and Histologic Grade, ranking the clinical severity specifically within each distinct cancer type.
* **KPI 6: Therapeutic Efficacy of Radiation Therapy:** I filter the cohort to isolate patients who strictly received or did not receive radiation therapy, evaluating its impact on survival months across different cancer types.

---

# Demographics

In [ ]:
-- Demographics 1: Top Cancer Types in the clinical cohort
SELECT 
    cancer_type, 
    COUNT(*) AS total_cases
FROM patients
GROUP BY cancer_type
ORDER BY total_cases DESC;


-- Demographics 2: Gender distribution by Cancer Type
SELECT 
    cancer_type, 
    sex, 
    COUNT(*) AS patient_count
FROM patients
WHERE sex IS NOT NULL
GROUP BY cancer_type, sex
ORDER BY cancer_type, patient_count DESC;


-- Demographics 3: Age distribution (Creating age buckets for better visualization)
SELECT 
    CASE 
        WHEN CAST(diagnosis_age AS NUMERIC) < 40 THEN '1. Under 40'
        WHEN CAST(diagnosis_age AS NUMERIC) BETWEEN 40 AND 59 THEN '2. 40-59'
        WHEN CAST(diagnosis_age AS NUMERIC) BETWEEN 60 AND 79 THEN '3. 60-79'
        ELSE '4. 80+'
    END AS age_group,
    COUNT(*) AS total_patients
FROM patients
WHERE diagnosis_age IS NOT NULL
GROUP BY age_group
ORDER BY age_group;

# KPIs:

In [ ]:
-- KPI 1: Difference in survival months based on activity levels
SELECT 
    CASE 
        WHEN vigorous_recreation = 1.0 THEN 'Vigorous Exercise'
        WHEN vigorous_recreation = 2.0 THEN 'Sedentary'
        ELSE 'Unknown'
    END AS activity_level,
    COUNT(*) AS total_patients,
    ROUND(AVG(CAST(survival_months AS NUMERIC)), 2) AS avg_survival_months
FROM nhanes_analytics_data
WHERE vigorous_recreation IN (1.0, 2.0)
GROUP BY activity_level
ORDER BY avg_survival_months DESC;


-- KPI 2: Risk Matrix mixing smoking with a sedentary lifestyle
SELECT 
    CASE 
        WHEN current_smoking_status IN (1.0, 2.0) THEN 'Smoker'
        WHEN current_smoking_status = 3.0 THEN 'Non-Smoker'
        ELSE 'Unknown'
    END AS smoking_status,
    CASE 
        WHEN vigorous_recreation = 1.0 THEN 'Active'
        WHEN vigorous_recreation = 2.0 THEN 'Sedentary'
        ELSE 'Unknown'
    END AS activity_level,
    COUNT(*) AS patient_count,
    ROUND(AVG(CAST(survival_months AS NUMERIC)), 2) AS avg_survival_months,
    RANK() OVER(ORDER BY AVG(CAST(survival_months AS NUMERIC)) DESC) AS survival_risk_rank
FROM nhanes_analytics_data
WHERE current_smoking_status IN (1.0, 2.0, 3.0) 
  AND vigorous_recreation IN (1.0, 2.0)
GROUP BY smoking_status, activity_level;


-- KPI 3: Impact of highly sedentary behavior 
SELECT 
    sedentary_category,
    COUNT(*) AS total_patients,
    ROUND(AVG(CAST(survival_months AS NUMERIC)), 2) AS avg_survival_months
FROM (
    SELECT 
        survival_months,
        CASE 
            WHEN CAST(sedentary_minutes_day AS NUMERIC) < 240 THEN 'Low Sedentary (<4 hours)'
            WHEN CAST(sedentary_minutes_day AS NUMERIC) BETWEEN 240 AND 480 THEN 'Moderate Sedentary (4-8 hours)'
            WHEN CAST(sedentary_minutes_day AS NUMERIC) > 480 THEN 'Highly Sedentary (>8 hours)'
            ELSE 'Unknown'
        END AS sedentary_category
    FROM nhanes_analytics_data
    WHERE sedentary_minutes_day IS NOT NULL
) AS categorized_data
GROUP BY sedentary_category
ORDER BY avg_survival_months DESC;

-- KPI 4: Impact of Genetic Mutation Burden on Disease-Free Survival (Cleaned)
-- I use a CTE (WITH) to pre-filter and exclude any 'Unknown' or unmeasured data
WITH categorized_mutations AS (
    SELECT 
        CASE 
            WHEN CAST(mutation_count AS NUMERIC) < 50 THEN 'Low Mutation Burden (<50)'
            WHEN CAST(mutation_count AS NUMERIC) BETWEEN 50 AND 150 THEN 'Moderate Mutation Burden (50-150)'
            WHEN CAST(mutation_count AS NUMERIC) > 150 THEN 'High Mutation Burden (>150)'
            ELSE 'Exclude' 
        END AS mutation_burden_category,
        disease_free_months,
        overall_survival_months
    FROM patients
    WHERE mutation_count IS NOT NULL 
      AND disease_free_months IS NOT NULL
)
SELECT 
    mutation_burden_category,
    COUNT(*) AS total_patients,
    ROUND(AVG(CAST(disease_free_months AS NUMERIC)), 2) AS avg_months_disease_free,
    ROUND(AVG(CAST(overall_survival_months AS NUMERIC)), 2) AS avg_months_overall_survival
FROM categorized_mutations
WHERE mutation_burden_category != 'Exclude' -- Removing the noise
GROUP BY mutation_burden_category
ORDER BY avg_months_disease_free DESC;


-- KPI 5: Clinical Severity Matrix by Cancer Type (Cleaned of 'Not Reported' / 'Unknown')
-- I added cancer_type to the partition to accurately compare severity within specific oncological pathologies
SELECT 
    cancer_type,
    neoplasm_disease_stage_american_joint_committee_on_cancer_code AS tumor_stage,
    neoplasm_histologic_grade AS histologic_grade,
    COUNT(*) AS patient_count,
    ROUND(AVG(CAST(overall_survival_months AS NUMERIC)), 2) AS avg_survival_months,
    DENSE_RANK() OVER(
        PARTITION BY cancer_type, neoplasm_disease_stage_american_joint_committee_on_cancer_code 
        ORDER BY AVG(CAST(overall_survival_months AS NUMERIC)) ASC
    ) AS survival_risk_rank_within_stage
FROM patients
WHERE overall_survival_months IS NOT NULL
  AND cancer_type IS NOT NULL
  AND neoplasm_disease_stage_american_joint_committee_on_cancer_code IS NOT NULL
  AND neoplasm_histologic_grade IS NOT NULL
  AND LOWER(neoplasm_disease_stage_american_joint_committee_on_cancer_code) NOT LIKE '%unknown%'
  AND LOWER(neoplasm_disease_stage_american_joint_committee_on_cancer_code) NOT LIKE '%not%'
  AND LOWER(neoplasm_histologic_grade) NOT LIKE '%unknown%'
  AND LOWER(neoplasm_histologic_grade) NOT LIKE '%not%'
GROUP BY cancer_type, tumor_stage, histologic_grade
ORDER BY cancer_type, tumor_stage, survival_risk_rank_within_stage;


-- KPI 6: Therapeutic Efficacy of Radiation Therapy by Cancer Type (Cleaned)
-- I strictly filter for only 'YES' or 'NO' answers, automatically discarding 'Unknowns'
SELECT 
    cancer_type,
    CASE 
        WHEN LOWER(radiation_therapy) = 'yes' THEN 'Received Radiation'
        WHEN LOWER(radiation_therapy) = 'no' THEN 'No Radiation'
    END AS therapy_status,
    COUNT(*) AS total_patients,
    ROUND(AVG(CAST(overall_survival_months AS NUMERIC)), 2) AS avg_survival_months
FROM patients
WHERE overall_survival_months IS NOT NULL 
  AND cancer_type IS NOT NULL
  AND LOWER(radiation_therapy) IN ('yes', 'no') -- This deletes the unknowns instantly
GROUP BY cancer_type, therapy_status
HAVING COUNT(*) >= 5
ORDER BY cancer_type, avg_survival_months DESC;